# Module 1 · Virtual Lab: from question to research strategy

**Swarm in the Lab** · BiocAsia 2026 · 13:35–14:25

A small team of AI scientists (a Principal Investigator, three specialists and a critic) meets to turn one question into a research strategy:

> **Is B7-H3 (CD276) a credible therapeutic target in lung cancer, and which modality fits it?**

| Time | Section | What you do |
|---|---|---|
| 13:35 | **1 · Walkthrough** | See how Virtual Lab builds agents and runs meetings, with Claude Code explaining the code |
| 13:50 | **2 · Hands-on** | Set the question, build the team, run a three-round team meeting |
| 14:10 | **3 · Hands-on** | Run more meetings in parallel, merge them, and pull out a research strategy |
| — | **⭐ Stretch** | Ground a meeting in PubMed and check its citations |

**What you leave with:** `research_strategy.json` and `research_strategy.md` in your outputs folder. Module 2 picks up the analyses it asks for, and in Module 3 you send it to the Virtual Biotech team.

**Pick your path**
- 🟢 **Beginner:** run the cells as they are. If you change one thing, make it the question in 2a.
- 🔵 **Core:** edit the agents, agenda and rules, and run every meeting live.
- 🟣 **Advanced:** finish with the ⭐ stretch task.

**Falling behind?** Set `USE_CACHE = True` in the next cell and re-run. Each meeting then loads a pre-run copy instantly and for free, so you can still read, compare and merge.

## 0 · Setup

In [ ]:
from pathlib import Path

WORKSHOP_DIR = Path.home() / "workshop"
OUTPUT_DIR   = WORKSHOP_DIR / "outputs" / "module1"      # your meetings and strategy are saved here
CACHE_DIR    = WORKSHOP_DIR / "cache" / "module1"        # pre-run meetings (organisers: see the end of this notebook)

MODEL = "gpt-5.2"     # Virtual Lab's default model; every agent below uses it
USE_CACHE = False     # True = load pre-run meetings instead of calling the model (instant, free)

# Your OpenAI key is read from the private key file set up in the self-test notebook (Step 0).
# Never paste a key into this notebook.

In [ ]:
# Imports and helper functions. Skim the comments; you don't need to read every line.
import concurrent.futures, datetime, hashlib, html, json, os, re, shutil
from IPython.display import display, Markdown, HTML

# --- Hiding secrets ------------------------------------------------------------------
# Anything that looks like a key or token is replaced before it's shown on screen or saved.
_SECRET_VALUES = set()
_SECRET_RE = re.compile(   # (?<!…) stops ordinary words like "risk-assessment" from matching
    r"(?<![A-Za-z0-9])(?:sk-[A-Za-z0-9_*.\-]{16,}|hf_[A-Za-z0-9]{20,}|gh[pousr]_[A-Za-z0-9]{20,}"
    r"|github_pat_[A-Za-z0-9_]{20,}|(?:AKIA|ASIA)[A-Z0-9]{16}\b"
    r"|eyJ[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,})")

def redact(text):
    s = "" if text is None else str(text)
    for value in _SECRET_VALUES:
        s = s.replace(value, "[hidden]")
    return _SECRET_RE.sub("[hidden]", s)

# --- Workshop keys --------------------------------------------------------------------
# Keys live in one file that only your user can read. They are loaded into this notebook's
# process only: not into your terminal, and not into Claude Code. They are never printed.
KEY_FILE = Path.home() / ".config" / "swarm-lab" / "keys.env"
KEY_NAMES = ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_BASE_URL", "ANTHROPIC_BASE_URL")

def _read_key_file():
    values = {}
    try:
        lines = KEY_FILE.read_text().splitlines()
    except OSError:
        return values
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        name = name.strip().removeprefix("export ").strip()
        value = value.strip().strip('"').strip("'")
        if name in KEY_NAMES and value:
            values[name] = value
    return values

def key_file_is_private():
    """True if only the owner can read the key file and its folder."""
    try:
        return (KEY_FILE.stat().st_mode & 0o077) == 0 and (KEY_FILE.parent.stat().st_mode & 0o077) == 0
    except OSError:
        return False

def load_keys(names):
    """Load the named keys into this notebook only. Returns where each came from:
    'environment', 'key file' or None (missing)."""
    from_file = _read_key_file()
    sources = {}
    for name in names:
        if os.environ.get(name):
            sources[name] = "environment"
        elif name in from_file:
            os.environ[name] = from_file[name]
            sources[name] = "key file"
        else:
            sources[name] = None
        if name.endswith("_KEY") and os.environ.get(name):
            _SECRET_VALUES.add(os.environ[name])
    return sources

def fingerprint(value):
    """A short label that tells keys apart without revealing them."""
    return hashlib.sha256(value.encode()).hexdigest()[:8]

# This module only needs the OpenAI key (Virtual Lab). The Anthropic key isn't loaded.
_KEY_SOURCES = load_keys(["OPENAI_API_KEY", "OPENAI_BASE_URL"])
if not os.environ.get("OPENAI_API_KEY") and not USE_CACHE:
    print("⚠️ No OpenAI key found. Run Step 0 of 00_preflight.ipynb, or set USE_CACHE = True to use pre-run meetings.")

# Virtual Lab counts tokens with tiktoken, which downloads its encoding once. Keep that download in the workshop
# folder (it survives reboots and is copied into every clone) instead of /tmp.
os.environ.setdefault("TIKTOKEN_CACHE_DIR", str(WORKSHOP_DIR / ".cache" / "tiktoken"))

import virtual_lab
from virtual_lab.agent import Agent
from virtual_lab.run_meeting import run_meeting
from virtual_lab.prompts import (PRINCIPAL_INVESTIGATOR, SCIENTIFIC_CRITIC,
                                 create_merge_prompt, team_meeting_start_prompt)
from virtual_lab.constants import CONSISTENT_TEMPERATURE, CREATIVE_TEMPERATURE
from virtual_lab.utils import load_summaries

# --- Being polite to PubMed ------------------------------------------------------
# Virtual Lab's PubMed tool (the stretch task) calls NCBI with no pause and no retry, and NCBI allows only about
# 3 requests a second from one address. A busy room shares that limit, and one "429 Too Many Requests" used to end
# the whole meeting. Space the requests out, retry when NCBI says "slow down", and skip an article that still
# can't be fetched instead of stopping the meeting.
import random, threading, time, types
import requests
import virtual_lab.utils as _vl_utils

_NCBI_LOCK, _NCBI_LAST = threading.Lock(), [0.0]

def _polite_get(url, *args, **kwargs):
    kwargs.setdefault("timeout", 60)
    for attempt in range(6):
        with _NCBI_LOCK:
            wait = 0.5 - (time.monotonic() - _NCBI_LAST[0])
            if wait > 0:
                time.sleep(wait)
            _NCBI_LAST[0] = time.monotonic()
        try:
            response = requests.get(url, *args, **kwargs)
        except (requests.ConnectionError, requests.Timeout):
            if attempt == 5:
                raise
        else:
            if response.status_code not in (429, 500, 502, 503, 504) or attempt == 5:
                return response
            retry_after = response.headers.get("Retry-After", "")
            if retry_after.isdigit():
                time.sleep(min(int(retry_after), 30))
                continue
        time.sleep(min(2 ** attempt, 20) + random.random())

_vl_utils.requests = types.SimpleNamespace(get=_polite_get)   # only Virtual Lab's PubMed calls use this

_get_article = _vl_utils.get_pubmed_central_article
def _get_article_or_skip(pmcid, abstract_only=False):
    try:
        return _get_article(pmcid=pmcid, abstract_only=abstract_only)
    except requests.RequestException:
        return None, None   # Virtual Lab skips articles it can't read
_vl_utils.get_pubmed_central_article = _get_article_or_skip

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# --- Meeting plans: who speaks when (no model calls) ---------------------------
def _plan_team(lead, members, rounds):
    steps = [(str(lead), "opens the meeting")]
    for r in range(1, rounds + 1):
        steps += [(str(m), f"round {r}") for m in members]
        steps.append((str(lead), f"synthesises round {r}" if r < rounds else "writes the summary"))
    return steps

def _plan_individual(member, rounds):
    steps = [(str(member), "answers the agenda")]
    for r in range(1, rounds + 1):
        steps += [("Scientific Critic", "critiques"),
                  (str(member), "revises" if r < rounds else "revises (final answer = the summary)")]
    return steps

def show_plan(kind, rounds, lead=None, members=(), member=None, copies=1):
    steps = _plan_team(lead, members, rounds) if kind == "team" else _plan_individual(member, rounds)
    n = len(steps)
    lo, hi = max(1, round(n * 10 / 60)), max(2, round(n * 25 / 60))
    head = f"{kind.title()} meeting · {rounds} round(s) · {n} model calls, one after another (about {lo}–{hi} min)"
    if copies > 1:
        head += f"\n× {copies} meetings at the same time: {n * copies} calls, but about the same wall-clock time as one"
    print(head)
    for i, (who, what) in enumerate(steps, 1):
        print(f"  {i:>2}. {who:<26} {what}")
    print("Every speaker sees the whole conversation so far, plus its own system prompt.")


# --- Run a meeting, or load the pre-run copy ------------------------------------
def _load_cached(name):
    src = CACHE_DIR / f"{name}.json"
    if not src.exists():
        raise FileNotFoundError(f"No pre-run copy of '{name}' in {CACHE_DIR}. Set USE_CACHE = False to run it live.")
    for ext in (".json", ".md"):
        if (CACHE_DIR / f"{name}{ext}").exists():
            shutil.copy(CACHE_DIR / f"{name}{ext}", OUTPUT_DIR / f"{name}{ext}")
    print(f"📦 Loaded the pre-run copy of '{name}' (it used the workshop's default question and team).")
    return load_summaries([OUTPUT_DIR / f"{name}.json"])[0]

FELL_BACK = []   # meetings that failed live and were replaced by the pre-run copy

def run_or_load(name, **meeting_args):
    """Run a Virtual Lab meeting and return its summary. Falls back to the pre-run copy if the call fails."""
    if USE_CACHE:
        return _load_cached(name)
    try:
        return run_meeting(save_dir=OUTPUT_DIR, save_name=name, return_summary=True, **meeting_args)
    except Exception as e:
        if (CACHE_DIR / f"{name}.json").exists():
            print(redact(f"⚠️ The meeting stopped ({type(e).__name__}: {str(e)[:150]}). Using the pre-run copy instead."))
            FELL_BACK.append(name)
            return _load_cached(name)
        raise


# --- Reading meetings ------------------------------------------------------------
def _esc(text):
    return html.escape(text or "")

def show_text(title, text):
    """A collapsible block of raw text."""
    display(HTML(f"<details><summary><b>{_esc(title)}</b></summary>"
                 f"<div style='white-space:pre-wrap;font-size:0.9em;margin:6px 0 0 12px'>{_esc(text)}</div></details>"))

def show_transcript(name):
    """Who said how much, then every turn as a collapsible block."""
    turns = json.loads((OUTPUT_DIR / f"{name}.json").read_text())
    stats = {}
    for t in turns:
        if t["agent"] == "User":
            continue
        s = stats.setdefault(t["agent"], {"turns": 0, "words": 0, "passes": 0})
        s["turns"] += 1
        s["words"] += len(t["message"].split())
        s["passes"] += t["message"].strip().lower().rstrip(".!") == "pass"
    rows = "".join(f"<tr><td>{_esc(a)}</td><td>{s['turns']}</td><td>{s['words']:,}</td><td>{s['passes']}</td></tr>"
                   for a, s in stats.items())
    display(HTML(f"<b>{_esc(name)}</b>: who contributed what"
                 f"<table><tr><th>Agent</th><th>Turns</th><th>Words</th><th>Passed</th></tr>{rows}</table>"))
    blocks = []
    for i, t in enumerate(turns):
        is_prompt = t["agent"] == "User"
        label = "📝 Prompt from Virtual Lab" if is_prompt else f"🧑‍🔬 {t['agent']}"
        style = "color:#777;" if is_prompt else ""
        blocks.append(f"<details><summary style='{style}'>{i:>2}. {_esc(label)} · {len(t['message'].split()):,} words</summary>"
                      f"<div style='white-space:pre-wrap;font-size:0.9em;margin:6px 0 10px 18px;{style}'>{_esc(t['message'])}</div></details>")
    display(HTML("".join(blocks)))

def section(summary, heading):
    """Pull one '### Heading' section out of a meeting summary."""
    m = re.search(rf"^#{{1,4}}\s*{re.escape(heading)}\b[^\n]*\n(.*?)(?=^#{{1,3}}\s|\Z)", summary or "", flags=re.M | re.S | re.I)
    return m.group(1).strip() if m else "(this summary has no such section)"

def compare(summaries, heading, labels=None, max_chars=2500):
    """Show the same section of several summaries side by side."""
    labels = labels or [f"Meeting {i}" for i in range(1, len(summaries) + 1)]
    cols = []
    for label, s in zip(labels, summaries):
        text = section(s, heading)
        text = text if len(text) <= max_chars else text[:max_chars] + " …"
        cols.append(f"<div style='flex:1;min-width:260px;border:1px solid #ccc;border-radius:6px;padding:8px'>"
                    f"<b>{_esc(label)}</b><div style='white-space:pre-wrap;font-size:0.88em'>{_esc(text)}</div></div>")
    display(HTML(f"<p><b>{_esc(heading)}</b>, side by side</p><div style='display:flex;gap:10px;flex-wrap:wrap'>{''.join(cols)}</div>"))

key = os.environ.get("OPENAI_API_KEY")
print(f"Virtual Lab {virtual_lab.__version__} · model {MODEL} · saving to {OUTPUT_DIR}"
      + (f" · OpenAI key from {_KEY_SOURCES['OPENAI_API_KEY']} (fingerprint {fingerprint(key)})" if key else "")
      + (" · USING PRE-RUN MEETINGS" if USE_CACHE else ""))
del key

## 1 · Walkthrough: how Virtual Lab works · 13:35–13:50

Virtual Lab rests on two ideas: **agents** and **meetings**.

### Agents are prompts
An agent is a model plus a four-sentence system prompt. Here are the two that ship with Virtual Lab:

In [ ]:
for agent in (PRINCIPAL_INVESTIGATOR, SCIENTIFIC_CRITIC):
    print(f"{agent.title}  [{agent.model}]\n  {agent.prompt}\n")

That's the whole agent: no memory and, by default, no tools. Everything else comes from the meeting it's in.

### Two kinds of meeting
- **Team meeting:** a team lead and members. The lead opens, each member speaks in turn, and the lead synthesises. That repeats for *N* rounds, then the lead writes a summary.
- **Individual meeting:** one agent and the built-in Scientific Critic. The agent answers, the critic critiques, and the agent revises.

### Memory is the summary
A meeting's last message is its summary. Later meetings receive earlier summaries through `summaries=`, and background documents through `contexts=`. Both are pasted into the first message, so you can always read exactly what an agent was given.

Here is the running order of a three-round team meeting with five agents. Nothing is sent to a model yet:

In [ ]:
show_plan("team", rounds=3, lead="Principal Investigator",
          members=["Specialist 1", "Specialist 2", "Specialist 3", "Scientific Critic"])

### 💬 Ask Claude Code
Run the next cell to see where Virtual Lab is installed. Then open the Claude Code panel (the spark icon in the top-right corner of the editor) and paste one of the questions below, replacing `<path>` with the printed folder. Codex works too.

1. *"The Python package virtual_lab is installed at `<path>`. Read run_meeting.py and prompts.py. In a team meeting with 3 rounds, who speaks in what order, and what exactly does each agent see when it is their turn?"*
2. *"In virtual_lab's run_meeting, what's the difference between the `summaries` and `contexts` arguments? Show me where each ends up in the prompt."*
3. *"Where does the Scientific Critic come from in an individual meeting, and how could I use my own critic instead?"*
4. *"Why would you run the same meeting several times at a high temperature and then merge them? Show me the merge prompt."*

<details><summary><b>Check your answers</b> (open after asking)</summary>

1. The lead opens, then each member in order, then the lead synthesises, for each round. The lead writes the summary at the end. Every agent receives the **entire conversation so far**; only its system prompt differs.
2. Both are pasted into the opening message: `contexts` as background ("Here is context for this meeting"), and `summaries` as earlier meetings ("Here are summaries of the previous meetings"). There's no hidden memory.
3. It's the `SCIENTIFIC_CRITIC` constant in prompts.py, hard-coded into individual meetings. To use your own critic, add it as a member of a team meeting, which is what we do below.
4. Each run is one sample of a noisy process. Several runs at a higher temperature explore different answers, and a low-temperature merge keeps the best parts and says where each came from (`MERGE_PROMPT` in prompts.py).
</details>

In [ ]:
print(Path(virtual_lab.__file__).parent)

## 2 · Hands-on: your question and your team · 13:50–14:10

### 2a · The question
🟢 To use your own target, change `GENE` and `DISEASE`. Everything downstream follows. (Pre-run copies only exist for the default question.)

In [ ]:
GENE    = "B7-H3 (CD276)"
DISEASE = "lung cancer"

BACKGROUND = (
    f"You are working on a research project to decide whether {GENE} is a credible therapeutic target in {DISEASE}, "
    "and which therapeutic modality (for example an antibody-drug conjugate, CAR-T cells, a naked or bispecific antibody, "
    "a radioligand or a small molecule) best fits it."
)

# Given to the team as context, so the plan uses tools they will really have in Modules 2 and 3.
AVAILABLE_TOOLS = """Later in the project the team will be able to use only these tools, so plan analyses they can run:
- Scanpy: quality control, clustering and marker-gene analysis of the workshop single-cell RNA-seq dataset: 20,000 cells from lung adenocarcinoma tumours (the single-cell lung cancer atlas, LuCA), labelled by cell type, including malignant cells
- TISSUE: spatial gene-expression prediction with calibrated uncertainty. The only spatial data available is TISSUE's small example dataset (mouse brain, 33 genes, not including the target), so it can demonstrate the method but not test the target
- CELLxGENE Census: single-cell expression across healthy human tissues
- Open Targets Platform: target-disease associations, genetic evidence, tractability and known safety liabilities
- ClinicalTrials.gov: registered clinical trials
- ChEMBL: compounds and bioactivity data
- PubMed: the literature"""

AGENDA = (
    f"{BACKGROUND} Design a research strategy that the team can carry out with public data and the tools in the context. "
    "The strategy must look for evidence against the target as well as for it."
)

AGENDA_QUESTIONS = (
    f"How strong is the current case that {GENE} is a credible therapeutic target in {DISEASE}, and what evidence would most change your view?",
    "Which therapeutic modality fits this target best, and what evidence would distinguish between the leading options?",
    "What are the main safety liabilities (for example, expression in healthy tissues), and how will the team check them?",
    "List 3-6 concrete analyses. For each, give the question it answers, the tool or database, the expected output, and the result that would count against the target.",
    "What result would make the team abandon this target?",
)

AGENDA_RULES = (
    "Only propose analyses that use the tools in the context or other public data.",
    "Do not cite specific papers, trial IDs or numbers unless they were given to you; say what should be looked up instead.",
    "Keep the strategy small enough to run in one afternoon.",
)
print(AGENDA)

### 2b · The team
Each field slots into a sentence: *"You are a {title}. Your expertise is in {expertise}. Your goal is to {goal}. Your role is to {role}."* So start `goal` and `role` with a verb.

🔵 Try sharpening one specialist, or swapping one out (an immunologist? a clinician?).

In [ ]:
pi = Agent(
    title="Principal Investigator",
    expertise="translational oncology and drug target validation",
    goal=f"decide whether {GENE} is worth pursuing as a therapeutic target in {DISEASE}, and how",
    role="lead the team, weigh the evidence for and against, make the final call, and keep the plan feasible",
    model=MODEL,
)
statistical_geneticist = Agent(
    title="Statistical Geneticist",
    expertise="human genetics and cancer genomics as evidence for drug targets, including somatic alterations, germline associations and Mendelian randomisation",
    goal="judge whether genetic and genomic evidence supports the target",
    role="propose analyses of genetic and genomic evidence, and say clearly when evidence is only correlational",
    model=MODEL,
)
single_cell_biologist = Agent(
    title="Single-Cell Biologist",
    expertise="single-cell and spatial transcriptomics of tumours and healthy tissues",
    goal="establish which cells express the target in tumours and in healthy tissues",
    role="design expression analyses (tumour versus normal, cell types, spatial context) using single-cell and spatial data",
    model=MODEL,
)
pharmacologist = Agent(
    title="Pharmacologist",
    expertise="oncology drug modalities (antibody-drug conjugates, cell therapies, bispecifics, radioligands and small molecules) and their safety",
    goal="match the target to the modality most likely to work safely",
    role="assess tractability, modality fit, on-target off-tumour toxicity and the clinical landscape",
    model=MODEL,
)
# The built-in critic, re-created so it uses the same model as the rest of the team.
critic = Agent(title=SCIENTIFIC_CRITIC.title, expertise=SCIENTIFIC_CRITIC.expertise,
               goal=SCIENTIFIC_CRITIC.goal, role=SCIENTIFIC_CRITIC.role, model=MODEL)

TEAM = (statistical_geneticist, single_cell_biologist, pharmacologist, critic)

for agent in (pi, *TEAM):
    print(f"• {agent.prompt}\n")

### 2c · Look before you run
This is the running order and the exact opening message the team will receive. It's worth thirty seconds: whatever isn't in this message, the agents don't know.

In [ ]:
NUM_ROUNDS = 3   # the plan for today; use 2 if you're short of time

show_plan("team", NUM_ROUNDS, lead=pi, members=TEAM)
show_text("Opening message every agent receives (click to expand)",
          team_meeting_start_prompt(team_lead=pi, team_members=TEAM, agenda=AGENDA,
                                    agenda_questions=AGENDA_QUESTIONS, agenda_rules=AGENDA_RULES,
                                    contexts=(AVAILABLE_TOOLS,), num_rounds=NUM_ROUNDS))

### 2d · Run the team meeting
This takes a few minutes and prints its cost and time when it finishes. While it runs, read the opening message above, or ask Claude Code one of the questions from Section 1.

In [ ]:
summary_1 = run_or_load(
    "team_meeting_1",
    meeting_type="team",
    team_lead=pi,
    team_members=TEAM,
    agenda=AGENDA,
    agenda_questions=AGENDA_QUESTIONS,
    agenda_rules=AGENDA_RULES,
    contexts=(AVAILABLE_TOOLS,),
    num_rounds=NUM_ROUNDS,
    temperature=CONSISTENT_TEMPERATURE,
)

### 2e · Read it
First the PI's summary, then the whole meeting turn by turn. As you read, look for:
- **The critic:** did it change anything, or just add caveats?
- **Passes and echoes:** did anyone say "pass" or simply agree?
- **The final call:** does the PI's recommendation follow the discussion, or override it? Is the reason given?
- **Invented specifics:** the rules said not to cite papers, trial IDs or numbers. Did any slip through?

In [ ]:
display(Markdown(summary_1))

In [ ]:
show_transcript("team_meeting_1")

## 3 · Hands-on: parallel meetings, merge, strategy · 14:10–14:25

### 3a · More meetings at once
One meeting is one sample. Virtual Lab's answer is to run the same meeting several times at a higher temperature, so the team explores different answers, and then have the PI merge the best parts. You already have meeting 1, so run two more **at the same time**, with two rounds each to fit the slot.

In [ ]:
N_EXTRA = 2          # extra meetings, run in parallel
EXTRA_ROUNDS = 2

show_plan("team", EXTRA_ROUNDS, lead=pi, members=TEAM, copies=N_EXTRA)

def extra_meeting(i):
    return run_or_load(
        f"team_meeting_{i}",
        meeting_type="team", team_lead=pi, team_members=TEAM,
        agenda=AGENDA, agenda_questions=AGENDA_QUESTIONS, agenda_rules=AGENDA_RULES,
        contexts=(AVAILABLE_TOOLS,), num_rounds=EXTRA_ROUNDS,
        temperature=CREATIVE_TEMPERATURE,   # more varied answers
    )

extra_summaries = []
with concurrent.futures.ThreadPoolExecutor(max_workers=N_EXTRA) as pool:
    futures = {pool.submit(extra_meeting, i): i for i in range(2, N_EXTRA + 2)}
    for future in concurrent.futures.as_completed(futures):
        try:
            extra_summaries.append((futures[future], future.result()))
        except Exception as e:
            print(f"⚠️ Meeting {futures[future]} failed ({type(e).__name__}); carrying on without it.")

summaries = (summary_1, *[s for _, s in sorted(extra_summaries)])
print(f"\nYou now have {len(summaries)} meeting summaries.")

### 3b · Compare, then merge
Put the recommendations side by side. Where the meetings agree, you can be a little more confident. Where they disagree, that's where you, the human, need to look hardest. Module 4 comes back to this.

In [ ]:
compare(summaries, "Recommendation")

In [ ]:
# Change the heading to compare another part of the summaries, e.g. "Answers" or "Next Steps".
compare(summaries, "Next Steps")

Now the PI merges them, in an individual meeting with one round of critique. Virtual Lab's merge prompt asks the PI to keep the best components **and say which meeting each came from**.

*Note: individual meetings use Virtual Lab's built-in critic, which runs on Virtual Lab's default model.*

In [ ]:
show_plan("individual", 1, member=pi)

merged = run_or_load(
    "merged",
    meeting_type="individual",
    team_member=pi,
    summaries=summaries,
    agenda=create_merge_prompt(agenda=AGENDA, agenda_questions=AGENDA_QUESTIONS, agenda_rules=AGENDA_RULES),
    contexts=(AVAILABLE_TOOLS,),
    num_rounds=1,
    temperature=CONSISTENT_TEMPERATURE,
)
display(Markdown(merged))

### 3c · From discussion to strategy
The merged plan is prose. Modules 2 and 3 need a list of analyses. One last structured call turns it into JSON. Note that this is the only thing passed between frameworks: **a document, not a shared memory.** Whatever this file leaves out, the next team won't know.

In [ ]:
TOOL_CHOICES = ["scanpy", "tissue", "cellxgene_census", "open_targets", "clinicaltrials_gov", "chembl", "pubmed", "other"]

EXTRACT_INSTRUCTIONS = f"""Convert the research plan you are given into JSON with exactly these keys:
- "question": one sentence
- "hypothesis": one sentence
- "recommended_modality": the modality, then a one-sentence reason
- "analyses": a list of 3-6 objects with keys "id" ("A1", "A2", ...), "question", "tool" (one of {TOOL_CHOICES}),
  "data", "expected_output" and "evidence_against" (the result that would count against the target)
- "safety_checks": a list of strings
- "kill_criteria": a list of strings
- "open_questions": a list of strings
Use only what the plan says. Do not add analyses, papers or numbers. If the plan doesn't say something, write "not specified"."""

def extract_strategy(plan_text):
    from openai import OpenAI
    response = OpenAI().chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": EXTRACT_INSTRUCTIONS},
                  {"role": "user", "content": plan_text}],
    )
    return json.loads(response.choices[0].message.content)

plan_text = merged if "merged" in globals() else summary_1
cached_strategy = CACHE_DIR / "research_strategy.json"
if USE_CACHE:
    strategy = json.loads(cached_strategy.read_text())
    print("📦 Loaded the pre-run strategy.")
else:
    try:
        strategy = extract_strategy(plan_text)
    except Exception as e:
        if not cached_strategy.exists():
            raise
        print(f"⚠️ Extraction failed ({type(e).__name__}). Using the pre-run strategy instead.")
        strategy = json.loads(cached_strategy.read_text())
        FELL_BACK.append("research_strategy")

# The model may return a string or null where a list belongs; make every list field a real list.
def _as_list(value):
    return [] if value is None else value if isinstance(value, list) else [value]
for key in ("safety_checks", "kill_criteria", "open_questions"):
    strategy[key] = [str(x) for x in _as_list(strategy.get(key))]
strategy["analyses"] = [a for a in _as_list(strategy.get("analyses")) if isinstance(a, dict)]

# Where will each analysis run this afternoon?
WHERE = {"scanpy": "Module 2 · paper agent", "tissue": "Module 2 · paper agent",
         "cellxgene_census": "Module 3 · database tool", "open_targets": "Module 3 · database tool",
         "clinicaltrials_gov": "Module 3 · database tool", "chembl": "Module 3 · database tool",
         "pubmed": "literature (Module 1 stretch)"}

print(f"Question:   {strategy.get('question')}\nHypothesis: {strategy.get('hypothesis')}\n"
      f"Modality:   {strategy.get('recommended_modality')}\n")
rows = "".join(
    f"<tr><td>{_esc(a.get('id'))}</td><td>{_esc(a.get('question'))}</td><td>{_esc(a.get('tool'))}</td>"
    f"<td>{_esc(WHERE.get(str(a.get('tool')).lower(), '⚠️ no tool for this today'))}</td>"
    f"<td>{_esc(a.get('evidence_against'))}</td></tr>"
    for a in strategy.get("analyses", []))
display(HTML("<table><tr><th>ID</th><th>Analysis</th><th>Tool</th><th>Runs in</th><th>Counts against the target if…</th></tr>"
             f"{rows}</table>"))

### 3d · Your call
The agents proposed; you decide. Drop any analysis you don't trust or can't run, add a note, then save. The file records that a human reviewed it.

An analysis marked **⚠️ no tool for this today** isn't a failure. The plan asked for something the afternoon's tools can't do, which is worth noting for Module 2.

In [ ]:
DROP  = []    # e.g. ["A4"] to remove an analysis before saving
NOTES = ""    # anything Module 2 and 3 should know, e.g. "A2 needs a spatial dataset we don't have"

final = dict(strategy)
final["analyses"] = [a for a in strategy.get("analyses", []) if a.get("id") not in DROP]
final["provenance"] = {
    "made_by": f"Virtual Lab {virtual_lab.__version__}, model {MODEL}",
    "meetings": sorted(p.stem for p in OUTPUT_DIR.glob("*.json") if p.stem.startswith(("team_meeting", "merged"))),
    "used_pre_run_copies": USE_CACHE or bool(FELL_BACK),
    "pre_run_copies_used_for": "all meetings" if USE_CACHE else FELL_BACK,
    "human_review": {"dropped": DROP, "notes": NOTES},
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
}

def strategy_markdown(s, plan):
    lines = [f"# Research strategy: {GENE} in {DISEASE}", "",
             f"**Question:** {s.get('question')}", "", f"**Hypothesis:** {s.get('hypothesis')}", "",
             f"**Recommended modality:** {s.get('recommended_modality')}", "", "## Analyses", ""]
    for a in s.get("analyses", []):
        lines += [f"### {a.get('id')} · {a.get('question')}",
                  f"- Tool: {a.get('tool')}", f"- Data: {a.get('data')}",
                  f"- Expected output: {a.get('expected_output')}",
                  f"- Counts against the target if: {a.get('evidence_against')}", ""]
    for key, title in (("safety_checks", "Safety checks"), ("kill_criteria", "Kill criteria"), ("open_questions", "Open questions")):
        lines += [f"## {title}", ""] + [f"- {x}" for x in s.get(key, [])] + [""]
    p = s["provenance"]
    lines += ["## Provenance", "", f"- Made by {p['made_by']} from meetings: {', '.join(p['meetings'])}",
              f"- Reviewed by a human. Dropped: {p['human_review']['dropped'] or 'none'}. Notes: {p['human_review']['notes'] or 'none'}",
              f"- Created {p['created']}", "", "## Appendix: the merged meeting summary", "", plan]
    return "\n".join(lines)

(OUTPUT_DIR / "research_strategy.json").write_text(json.dumps(final, indent=2))
(OUTPUT_DIR / "research_strategy.md").write_text(strategy_markdown(final, plan_text))
print(f"✅ Saved {OUTPUT_DIR / 'research_strategy.json'}\n✅ Saved {OUTPUT_DIR / 'research_strategy.md'}\n")
print(f"{len(final['analyses'])} analyses go to Modules 2 and 3. That's the end of Module 1. Enjoy the break!")

## ⭐ Stretch: ground a meeting in PubMed
So far the agents have worked from memory, and the rules told them not to cite anything. Virtual Lab can instead give agents a PubMed Central search tool (`pubmed_search=True`). The agent decides when to search, and the search results are pasted into the conversation.

Then we check the citations: which articles were actually retrieved, and which were cited. A citation of an article the tool never returned is a red flag, and a preview of Module 4.

In [ ]:
PUBMED_AGENDA = (
    f"{BACKGROUND} Search PubMed Central and summarise the most important evidence on (1) {GENE} expression "
    f"in {DISEASE} tumours versus healthy tissues and (2) clinical results of {GENE}-targeted therapies. "
    "Cite every claim with the PMCID of an article you retrieved."
)
PUBMED_RULES = (
    "When you search, set abstract_only to true and ask for at most 3 articles per search.",
    "Only cite PMCIDs that appeared in your own search results.",
)

show_plan("individual", 1, member=single_cell_biologist)
pubmed_summary = run_or_load(
    "pubmed_grounded",
    meeting_type="individual",
    team_member=single_cell_biologist,
    agenda=PUBMED_AGENDA,
    agenda_rules=PUBMED_RULES,
    num_rounds=1,
    pubmed_search=True,
    temperature=CONSISTENT_TEMPERATURE,
)
display(Markdown(pubmed_summary))

In [ ]:
def check_citations(name):
    turns = json.loads((OUTPUT_DIR / f"{name}.json").read_text())
    retrieved = set()
    for t in turns:
        if t["agent"] == "Tool":
            retrieved |= set(re.findall(r"PMCID = (\d+)", t["message"]))
    final_answer = turns[-1]["message"]
    cited = set(re.findall(r"PMC(?:ID)?[\s:=#]*(\d{5,9})", final_answer, flags=re.I))
    print(f"Articles the tool retrieved: {len(retrieved)}  {sorted(retrieved)}")
    print(f"PMCIDs cited in the answer:  {len(cited)}  {sorted(cited)}\n")
    ghosts = cited - retrieved
    if ghosts:
        print(f"❌ Cited but never retrieved: {sorted(ghosts)}. Look these up yourself before trusting them.")
    else:
        print("✅ Every cited PMCID was retrieved by the tool.")
    if retrieved - cited:
        print(f"ℹ️ Retrieved but not cited: {sorted(retrieved - cited)}. What did the agent leave out, and why?")
    print("\nA retrieved citation still isn't a checked claim: open one article and check that it says what the answer says it does.")

check_citations("pubmed_grounded")

**More stretch ideas**
- Change `GENE` and `DISEASE` in 2a to a target from your own work, and re-run Sections 2 and 3.
- Give the PubMed tool to the whole team: add `pubmed_search=True` to the team meeting in 2d. Does the plan change? What happens to the cost?
- Replace the critic in `TEAM` with a sceptical clinician, and compare the recommendations.

---
### For organisers
- **Build the pre-run copies on your instance, before the image is made:** with the default question, set `USE_CACHE = False` and run the whole notebook, including the stretch. Then copy everything in `OUTPUT_DIR` to `CACHE_DIR`. Every clone inherits the copies, and any meeting that fails on the day falls back to them automatically. `prepare_image.py` then empties `OUTPUT_DIR`, so participants start fresh.
- **The "completed notebook" for the break:** after that run, save a copy with its outputs to `~/workshop/completed/`. `prepare_image.py` clears outputs only from notebooks in the top level of `~/workshop`.
- **Python must be 3.12.** Virtual Lab 1.2.0 needs 3.12 or later (Python 3.11 can't parse its f-strings), and CELLxGENE Census supports up to 3.12.
- **Keys:** this notebook loads only the OpenAI key, and only into its own process. It never prints a key: only a fingerprint.
- **Cost and load:** with the defaults, each participant makes about 42 model calls (16 for the first meeting, 22 for the two parallel meetings, 3 for the merge and 1 for the extraction), plus 3–6 for the stretch. Section 3a sends two meetings at once, so a full room puts real pressure on your OpenAI organisation's rate limits. Run the notebook once to see the real cost, which `run_meeting` prints after each meeting.